# SENTINEL — YOLOv8s Weapon Training on Kaggle (Phase 3, dataset v2)

Colab wali notebook ka Kaggle version: dataset v2 (train 21,809) par 100-epoch retrain, **2x T4** GPUs par (~7 ghante, aik session mein).

**Pehle ye 3 cheezein set karo (notebook settings, right sidebar):**
1. **Accelerator: GPU T4 x2** select karo (P100 nahi — wo 1 GPU hai, training ~13 ghante ho jayegi).
2. **Internet: ON** rakho (pip install + repo clone + weights download ke liye).
3. **Add Input → dataset `sentinel-weapon-v2`** attach karo (private dataset, Muse ne upload kiya hai).

**Phir: Save & Run All** dabao — poori notebook apne session mein chalegi (12h cap, idle timeout nahi lagta). Kisi cell mai red error aye to **wahin ruk jao** aur output Muse ko bhejo, khud fix mat karo.

Training script repo se ata hai (`scripts/training/train_weapon_detector.py`) — notebook use clone karke chalati hai, taake Kaggle aur repo hamesha ek hi code use karein.

In [ ]:
# ============ CONFIG — sirf yahi cell badalna hai ============
RUN_MODE = "v2"        # "v2" = 100 epochs (dataset v2 retrain) | "smoke" = 10 epochs (pipeline check)
REPO_BRANCH = "main"   # training script isi branch se clone hota hai

from datetime import datetime
from pathlib import Path

EPOCHS = 10 if RUN_MODE == "smoke" else 100 if RUN_MODE == "v2" else 50
RUN_NAME = f"weapon_yolov8s_{RUN_MODE}_{datetime.now():%Y%m%d_%H%M}"

DATASET_ZIP = Path("/kaggle/input/sentinel-weapon-v2/weapon_yolo_v2.zip")
WORK = Path("/kaggle/working/datasets")
RUNS_PROJECT = "/kaggle/working/runs/weapon"
MODEL_PATH = "/kaggle/working/yolov8s.pt"
OUTPUT_DIR = Path("/kaggle/working/output") / RUN_NAME   # best.pt + results yahan — run ke baad download kar lena

print("RUN_MODE :", RUN_MODE, "| EPOCHS:", EPOCHS)
print("RUN_NAME :", RUN_NAME)

## Step 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU nahi mila — notebook settings mai Accelerator = GPU T4 x2 select karo, phir dobara chalao."
N_GPU = torch.cuda.device_count()
print("GPUs:", N_GPU, "|", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
# 2 GPUs hon to DDP (device 0,1, batch 32); 1 GPU ho to single-GPU (device 0, batch 16, ~13 ghante)
DEVICE = "0,1" if N_GPU >= 2 else "0"
BATCH = 32 if N_GPU >= 2 else 16
print("DEVICE:", DEVICE, "| BATCH:", BATCH)

## Step 2 — Dataset zip ki mojoodgi (Add Input se attach hona chahiye)

In [ ]:
assert DATASET_ZIP.exists(), f"Dataset zip nahi mila: {DATASET_ZIP} — right sidebar mai Add Input se dataset 'sentinel-weapon-v2' attach karo, phir dobara chalao."
print("Dataset zip mila:", DATASET_ZIP.name, "| size (MB):", round(DATASET_ZIP.stat().st_size / 1e6, 1))

## Step 3 — Ultralytics install + model weights ki mazboot download
GitHub 504 ka sabak: `wget -c` resume ke saath download hogi, fail ho to yehi cell dobara chalana — download wahi se resume hogi.

In [ ]:
%pip install -q ultralytics
!wget -c --tries=10 --waitretry=5 "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8s.pt" -O /kaggle/working/yolov8s.pt

size = Path(MODEL_PATH).stat().st_size
print("yolov8s.pt size (MB):", round(size / 1e6, 2))
assert size > 20_000_000, "Weights file adhoori lagti hai — ye cell dobara chalao (resume hogi)."

## Step 4 — Repo clone (training script wahi se ayega)

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/sentinel_repo
!git clone -b $REPO_BRANCH --single-branch https://github.com/muddassir1935/FYP.git /kaggle/working/sentinel_repo

script = Path("/kaggle/working/sentinel_repo/scripts/training/train_weapon_detector.py")
assert script.exists(), "Training script nahi mila — branch ka naam CONFIG mai check karo, Muse ko batao."
print("Training script ready:", script)

## Step 5 — Dataset unzip + verification (counts match na hon to training yahi ruk jayegi)

In [ ]:
import zipfile

WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP) as z:
    z.extractall(WORK)

# dataset root = wo folder jiske andar images/ aur labels/ dono hon (zip ki structure dono surton mai handle hai)
cands = [p.parent for p in WORK.rglob("images") if (p.parent / "labels").exists()]
assert cands, "Dataset structure samajh nahi aya — ye output Muse ko bhejo."
root = sorted(cands, key=lambda p: len(p.parts))[0]
print("Dataset root:", root)

IMG_EXTS = {".jpg", ".jpeg", ".png"}
expected = {"train": 21809, "val": 1351, "test": 680}
ok = True
for split, exp in expected.items():
    imgs = [p for p in (root / "images" / split).glob("*") if p.suffix.lower() in IMG_EXTS]
    lbls = list((root / "labels" / split).glob("*.txt"))
    print(f"{split}: images = {len(imgs)} (expected {exp}), labels = {len(lbls)}")
    if len(imgs) != exp:
        ok = False

# labels sanity: har line ki pehli value (class id) sirf 0 (Gun) ya 1 (Knife) ho
bad, checked = 0, 0
for txt in (root / "labels").rglob("*.txt"):
    for line in txt.read_text().splitlines():
        if line.strip():
            checked += 1
            if int(float(line.split()[0])) not in (0, 1):
                bad += 1
print(f"label lines checked = {checked}, bad class ids = {bad}")

assert ok and bad == 0, "DATASET VERIFICATION FAILED — training roki gayi. Ye poora output Muse ko bhejo."

# Kaggle ke liye data yaml (repo wali yaml local paths ke liye hai, ye wali /kaggle/working ke absolute path ke saath)
DATA_YAML = root / "sentinel_data.yaml"
DATA_YAML.write_text(f"path: {root}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 2\nnames: ['Gun', 'Knife']\n")
print("DATA_YAML:", DATA_YAML)
print("DATASET VERIFIED OK")

## Step 6 — Training (v2 100 epochs, 2x T4 par andazan ~7 ghante — Save & Run All se chalao, session 12h cap hai)

In [ ]:
%cd /kaggle/working/sentinel_repo
!python scripts/training/train_weapon_detector.py --data $DATA_YAML --model $MODEL_PATH --epochs $EPOCHS --imgsz 640 --batch $BATCH --device $DEVICE --workers 4 --patience 20 --project $RUNS_PROJECT --name $RUN_NAME

## Step 7 — Results (test split per final numbers + confusion matrix)

In [ ]:
import json
from IPython.display import Image, display

run_dir = Path(RUNS_PROJECT) / RUN_NAME
summary = json.loads((run_dir / "run_summary.json").read_text())
print("TEST METRICS:", json.dumps(summary["test_metrics"], indent=2))

cm = run_dir / "confusion_matrix.png"
if cm.exists():
    display(Image(str(cm), width=640))

## Step 8 — Weights + results output folder mein save (run ke baad download kar lena + COPY block Muse ko bhejna hai)

In [ ]:
import hashlib
import shutil

dest = OUTPUT_DIR
dest.mkdir(parents=True, exist_ok=True)

best = run_dir / "weights" / "best.pt"
dated = dest / f"{RUN_NAME}_best.pt"
shutil.copy(best, dated)
for extra in ["results.csv", "confusion_matrix.png", "results.png", "run_summary.json"]:
    srcp = run_dir / extra
    if srcp.exists():
        shutil.copy(srcp, dest / extra)
# last.pt bhi rakh lo — agar kabhi resume karna pare to kaam ayega
last = run_dir / "weights" / "last.pt"
if last.exists():
    shutil.copy(last, dest / f"{RUN_NAME}_last.pt")

sha = hashlib.sha256(dated.read_bytes()).hexdigest()
tm = summary["test_metrics"]
print("=" * 60)
print("===== COPY FROM HERE — ye block Muse ko bhej dena =====")
print(f"RUN OK — {RUN_NAME}")
print(f"Weights file : {dated.name}")
print(f"Size (MB)    : {dated.stat().st_size / 1e6:.2f}")
print(f"SHA-256      : {sha}")
print(f"Test mAP@0.5 : {tm.get('mAP50')} | mAP@0.5:0.95 : {tm.get('mAP50_95')} | P : {tm.get('precision')} | R : {tm.get('recall')} | F1 : {tm.get('f1')}")
for cls, vals in tm.get("per_class", {}).items():
    print(f"  {cls}: AP50 = {vals.get('AP50')}")
print("===== COPY TO HERE =====")
print("=" * 60)
print(f"Weights + results saved in: {dest}  (Kaggle Output tab se download kar lo)")